
================================================================

**Author:** Carlos Stalin Saritama Atopo  
**Email:** cssaritama@gmail.com  
**Area of Specialization:** Analytics, Advanced Analytics, and Artificial Intelligence  
**Date:** August 2025 

===============================================================


# 02 — Data Preprocessing & Feature Engineering

**Goals**
- Load raw OHLCV and build a unified panel
- Engineer **20+ features** (technicals, volatility, momentum, dummies)
- Prepare train/validation/test splits
- Perform basic EDA (correlations, distributions)

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10,5)

RAW_DIR = Path('../data/raw')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

raw_path = RAW_DIR / 'ohlcv_synthetic.parquet'
df = pd.read_parquet(raw_path)
df = df.sort_values(['ticker','date']).reset_index(drop=True)

df['ret_1d'] = df.groupby('ticker')['close'].pct_change()


## Technical Indicators and Rolling Features

In [ ]:
def add_features(g):
    g = g.copy()
    g['log_ret'] = np.log(g['close']).diff()
    for w in [5,10,20,50]:
        g[f'sma_{w}'] = g['close'].rolling(w).mean()
        g[f'ema_{w}'] = g['close'].ewm(span=w, adjust=False).mean()
        g[f'vol_{w}'] = g['ret_1d'].rolling(w).std()
        g[f'mom_{w}'] = g['close'] / g['close'].shift(w) - 1
    # RSI
    delta = g['close'].diff()
    up = delta.clip(lower=0).rolling(14).mean()
    down = -delta.clip(upper=0).rolling(14).mean()
    rs = up / (down + 1e-9)
    g['rsi_14'] = 100 - (100/(1+rs))
    # MACD
    ema12 = g['close'].ewm(span=12, adjust=False).mean()
    ema26 = g['close'].ewm(span=26, adjust=False).mean()
    g['macd'] = ema12 - ema26
    g['macd_signal'] = g['macd'].ewm(span=9, adjust=False).mean()
    # Bollinger
    bb_mid = g['close'].rolling(20).mean()
    bb_std = g['close'].rolling(20).std()
    g['bb_hi'] = bb_mid + 2*bb_std
    g['bb_lo'] = bb_mid - 2*bb_std
    g['bb_w'] = (g['bb_hi'] - g['bb_lo']) / (g['close']+1e-9)
    # ATR (approx)
    tr = (g['high']-g['low']).abs()
    g['atr_14'] = tr.rolling(14).mean()
    # Dummies
    g['dow'] = pd.to_datetime(g['date']).dt.dayofweek
    g = pd.get_dummies(g, columns=['dow'], prefix='dow', drop_first=False)
    # Target: next-day return sign
    g['target'] = (g['ret_1d'].shift(-1) > 0).astype(int)
    return g

feat = df.groupby('ticker', group_keys=False).apply(add_features)
feat = feat.dropna().reset_index(drop=True)
feat_path = PROC_DIR / 'features.parquet'
feat.to_parquet(feat_path, index=False)
feat.shape

## EDA: Correlations and Distributions

In [ ]:
num_cols = [c for c in feat.columns if feat[c].dtype!='O' and c not in ['target']]
subset = feat[num_cols].select_dtypes(include=[np.number]).dropna().sample(5000, random_state=42)

corr = subset.corr(numeric_only=True)
ax = corr['ret_1d'].sort_values(ascending=False).head(20).plot(kind='bar', title='Top correlations vs ret_1d')
plt.show()

feat['target'].value_counts(normalize=True)

## Train / Validation / Test Split (chronological)

In [ ]:
feat = feat.sort_values(['ticker','date'])
cut1 = int(len(feat)*0.6)
cut2 = int(len(feat)*0.8)
train, valid, test = feat.iloc[:cut1], feat.iloc[cut1:cut2], feat.iloc[cut2:]

for name,part in [('train',train),('valid',valid),('test',test)]:
    part.to_parquet(PROC_DIR / f'{name}.parquet', index=False)

len(train), len(valid), len(test)